# 05 - ¿Sirvió el fine-tuning? El afinado contra el modelo base con cuatro prompts

El notebook 03 comparó el modelo base con el afinado **solo en forma**. La calidad
del contenido la midió un juez, pero contra el **maestro** (`gpt-4o-mini`), nunca
contra el base. Eso dejaba sin responder la pregunta central del proyecto:
**¿sirvió entrenar?**

«Corre en local, sin API y sin costo» no justifica el fine-tuning: justifica usar
un modelo local, y el Qwen base también corre en local. Lo que justifica el
fine-tuning es superar al base **con el mejor prompt que se le pueda dar**. Por eso
se lo compara con cuatro prompts distintos:

| Variante | Instrucción | Esquema JSON | Reglas del maestro | Ejemplos | Qué pone a prueba |
|---|---|---|---|---|---|
| **afinado** | corta | aprendido | no | no | — |
| base + few-shot | corta | implícito en los ejemplos | no | 3 | la línea de base clásica |
| base A | corta | no | no | no | lo que aprendió el fine-tuning: recibe **el mismo prompt** que el afinado |
| base B | corta | **sí** | no | no | el prompt mínimo con el que el base sabe qué claves usar: la prueba más dura para el argumento del prompt corto |
| base C | la del maestro | **sí** | **sí** | no | exactamente las instrucciones que recibió `gpt-4o-mini` al generar el dataset: lo que se usaría sin entrenar |

## Diseño

| Decisión | Por qué |
|---|---|
| Los **mismos 150 fragmentos** que juzgó el notebook 03 (`random_state=13` sobre el test) | comparable con sus cifras, y nunca vistos en el entrenamiento |
| La **misma rúbrica** y el mismo juez (`gpt-4o`, temperatura 0) | que la diferencia sea del modelo, no de la vara |
| Decodificación **greedy** | que la diferencia no sea azar de muestreo |
| El few-shot usa los **mismos 3 ejemplos** que el notebook 03 | comparable con lo medido antes |
| **Todo se genera y se juzga en esta corrida**, incluido el maestro | seis fuentes, un juez, los mismos fragmentos; el notebook no depende de resultados guardados por otro |
| **Prueba pareada** (McNemar) | todos responden sobre los mismos fragmentos: se compara fragmento a fragmento |

La sección B hace el control de formato bajo muestreo que le faltó al notebook 04.

*Reemplaza una versión anterior de este notebook, que solo comparaba el afinado
contra el base con few-shot.*


In [1]:
import json, re, time, unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA = Path("../data")
ADAPTADOR = Path("../app/adapter")
MODELO = "Qwen/Qwen3-4B-Instruct-2507"
SEMILLA = 42
LOTE = 8
MAX_NEW = 300
torch.manual_seed(SEMILLA)

# Identica a la del entrenamiento (notebook 03) y a la de la aplicacion.
INSTRUCCION = (
    "Eres un docente de medicina. A partir del FRAGMENTO escribe UNA pregunta "
    "de opcion multiple en espanol neutro. Responde SOLO con JSON."
)
# El prompt con el que gpt-4o-mini genero el dataset (notebook 02).
MAESTRO = json.load(open("../scripts/prompt_v2.json", encoding="utf-8"))["prompt"]

train_df = pd.read_parquet(DATA / "mcq_train.parquet")
test_df = pd.read_parquet(DATA / "mcq_test.parquet")

tok = AutoTokenizer.from_pretrained(MODELO)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"            # obligatorio para generar por lotes

base = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(base, ADAPTADOR)
model.eval()
print(f"train {len(train_df):,} | test {len(test_df):,}")
print(f"GPU: {torch.cuda.get_device_name(0)} | memoria usada {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("un solo modelo en memoria: el base se obtiene apagando el LoRA con disable_adapter()")


W0910 20:57:55.746000 21436 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

train 3,614 | test 436
GPU: NVIDIA GeForce RTX 5070 Ti | memoria usada 8.32 GB
un solo modelo en memoria: el base se obtiene apagando el LoRA con disable_adapter()


In [2]:
def parsear(t):
    if not t:
        return None
    t = t.strip()
    if t.startswith("```"):
        t = t.strip("`").removeprefix("json").strip()
    try:
        d = json.loads(t)
        return d if isinstance(d, dict) else None
    except Exception:
        return None


def estructura_ok(d):
    return bool(d and d.get("pregunta") and d.get("correcta")
                and isinstance(d.get("incorrectas"), list) and len(d["incorrectas"]) == 3)


def sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s).lower())
                   if unicodedata.category(c) != "Mn")


# Variante B: la instruccion corta mas el esquema, sin reglas ni ejemplos.
ESQUEMA = (INSTRUCCION + " Usa exactamente esta forma:\n"
           '{"apto": true, "pregunta": "...", "correcta": "...", '
           '"incorrectas": ["...", "...", "..."], "dificultad": "facil|media|dificil"}')

# Los mismos 3 ejemplos que recibio el base en el notebook 03.
ejemplos = train_df.sample(3, random_state=SEMILLA)
FEWSHOT = []
for _, e in ejemplos.iterrows():
    FEWSHOT.append({"role": "user", "content": "FRAGMENTO:\n" + e["chunk_text"]})
    FEWSHOT.append({"role": "assistant", "content": json.dumps({
        "apto": True, "pregunta": e["pregunta"], "correcta": e["correcta"],
        "incorrectas": list(e["incorrectas"]), "dificultad": e["dificultad"]},
        ensure_ascii=False)})


def chat(sistema, fragmento, previos=()):
    msgs = [{"role": "system", "content": sistema}, *previos,
            {"role": "user", "content": "FRAGMENTO:\n" + fragmento}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


VARIANTES = {   # clave: (etiqueta, usa el modelo base, constructor del prompt)
    "afinado": ("afinado (LoRA)", False, lambda f: chat(INSTRUCCION, f)),
    "fewshot": ("base + few-shot", True, lambda f: chat(INSTRUCCION, f, FEWSHOT)),
    "A": ("base A: mismo prompt", True, lambda f: chat(INSTRUCCION, f)),
    "B": ("base B: + esquema", True, lambda f: chat(ESQUEMA, f)),
    "C": ("base C: prompt del maestro", True, lambda f: chat(MAESTRO, f)),
}


def generar(prompts, usar_base, **muestreo):
    """Genera por lotes. Devuelve textos, tokens de prompt reales y truncadas."""
    kw = dict(max_new_tokens=MAX_NEW, pad_token_id=tok.pad_token_id)
    kw.update(muestreo or dict(do_sample=False))
    textos, tokens_prompt, truncadas = [], [], 0
    for i in range(0, len(prompts), LOTE):
        ids = tok(prompts[i:i + LOTE], return_tensors="pt", padding=True).to(model.device)
        n = ids.input_ids.shape[1]
        with torch.no_grad():
            if usar_base:
                with model.disable_adapter():
                    out = model.generate(**ids, **kw)
            else:
                out = model.generate(**ids, **kw)
        for fila in out:
            nuevos = fila[n:]
            if int((nuevos != tok.pad_token_id).sum()) >= MAX_NEW:
                truncadas += 1           # llego al tope sin cerrar
            textos.append(tok.decode(nuevos, skip_special_tokens=True))
        tokens_prompt += ids.attention_mask.sum(1).tolist()   # sin contar el relleno
    return textos, tokens_prompt, truncadas

print("ejemplos de few-shot:", ejemplos["question_focus"].tolist())
print("variantes:", [v[0] for v in VARIANTES.values()])


ejemplos de few-shot: ['Cone-rod dystrophy', 'Cephalic Disorders', 'nonbullous congenital ichthyosiform erythroderma']
variantes: ['afinado (LoRA)', 'base + few-shot', 'base A: mismo prompt', 'base B: + esquema', 'base C: prompt del maestro']


## A. Generación

Las cinco variantes generan sobre los mismos 150 fragmentos, todas con greedy.


In [3]:
casos = test_df.sample(150, random_state=13).reset_index(drop=True)
fragmentos = casos["chunk_text"].tolist()

salidas, dicts, filas_forma = {}, {}, []
for clave, (etiqueta, usar_base, construir) in VARIANTES.items():
    t0 = time.time()
    textos, toks, _ = generar([construir(f) for f in fragmentos], usar_base)
    seg = (time.time() - t0) / len(fragmentos)
    ds = [parsear(t) for t in textos]
    salidas[clave], dicts[clave] = textos, ds
    filas_forma.append({
        "variante": etiqueta,
        "tokens de prompt": round(sum(toks) / len(toks)),
        "JSON valido": f"{sum(d is not None for d in ds)}/150",
        "estructura completa": f"{sum(estructura_ok(d) for d in ds)}/150",
        "descarto el fragmento (apto=false)": sum(1 for d in ds if d and d.get("apto") is False),
        "segundos por pregunta (lotes de 8)": round(seg, 2)})
    print(f"  {etiqueta}: listo en {seg * len(fragmentos) / 60:.1f} min", flush=True)

forma = pd.DataFrame(filas_forma).set_index("variante")
print()
print(forma.to_string())

claves_A = Counter(k for d in dicts["A"] if d for k in d.keys())
print("\nclaves que usa la variante A, que no conoce el esquema:", claves_A.most_common(8))


  afinado (LoRA): listo en 2.4 min


  base + few-shot: listo en 1.9 min


  base A: mismo prompt: listo en 2.1 min


  base B: + esquema: listo en 2.1 min


  base C: prompt del maestro: listo en 1.9 min



                            tokens de prompt JSON valido estructura completa  descarto el fragmento (apto=false)  segundos por pregunta (lotes de 8)
variante                                                                                                                                            
afinado (LoRA)                           137     150/150             150/150                                   0                                0.97
base + few-shot                          695     150/150             146/150                                   0                                0.78
base A: mismo prompt                     137     150/150               0/150                                   0                                0.83
base B: + esquema                        187     150/150             143/150                                   0                                0.85
base C: prompt del maestro               748     143/150             125/150                             

In [4]:
try:
    from dotenv import load_dotenv
    load_dotenv("../.env")
except Exception:
    pass
from openai import OpenAI

client = OpenAI()
MODELO_JUEZ = "gpt-4o"

# La MISMA rubrica del notebook 03, palabra por palabra.
JUEZ = """Eres un revisor de examenes de medicina. Recibes un FRAGMENTO, el TEMA
al que pertenece, y una PREGUNTA de opcion multiple construida a partir de el.

Evalua con severidad:
1. respaldo: la opcion correcta esta afirmada explicitamente en el fragmento?
2. distractor_verdadero: alguna incorrecta es TAMBIEN cierta segun el fragmento?
3. unica_respuesta: hay exactamente una respuesta defendible?
4. tema_correcto: la pregunta se refiere al TEMA indicado y no a otra enfermedad?

Responde SOLO con JSON:
{"respaldo": true/false, "distractor_verdadero": true/false,
 "unica_respuesta": true/false, "tema_correcto": true/false,
 "comentario": "una frase solo si algo falla"}"""

ESTRUCTURA_ROTA = {"_estructura_rota": True}   # no se juzga: cuenta como fallo


def juzgar(args):
    idx, quien, fila, d = args
    if not estructura_ok(d):
        return idx, quien, ESTRUCTURA_ROTA, None
    texto = (f"TEMA: {fila['question_focus']}\n\nFRAGMENTO:\n{fila['chunk_text']}\n\n"
             f"PREGUNTA: {d['pregunta']}\nCORRECTA: {d['correcta']}\n"
             + "\n".join(f"INCORRECTA: {x}" for x in d["incorrectas"]))
    motivo = None
    for intento in range(5):
        try:
            r = client.chat.completions.create(
                model=MODELO_JUEZ, temperature=0,
                messages=[{"role": "system", "content": JUEZ},
                          {"role": "user", "content": texto}])
            v = parsear(r.choices[0].message.content)
            if v is not None:
                return idx, quien, v, None
            motivo = "respuesta no parseable"
        except Exception as exc:
            motivo = type(exc).__name__
        time.sleep(2 ** intento)                  # 1, 2, 4, 8, 16 segundos
    return idx, quien, None, motivo              # tras 5 intentos: se excluye


QUIENES = ["maestro"] + list(VARIANTES)
tareas = []
for i, fila in casos.iterrows():
    tareas.append((i, "maestro", fila, {"pregunta": fila["pregunta"], "correcta": fila["correcta"],
                                        "incorrectas": list(fila["incorrectas"])}))
    for clave in VARIANTES:
        tareas.append((i, clave, fila, dicts[clave][i]))

a_juzgar = sum(1 for t in tareas if estructura_ok(t[3]))
print(f"{len(tareas)} preguntas, {a_juzgar} con estructura valida para el juez...", flush=True)
veredictos = {q: {} for q in QUIENES}
motivos = Counter()
with ThreadPoolExecutor(6) as pool:
    for fut in as_completed([pool.submit(juzgar, t) for t in tareas]):
        idx, quien, v, motivo = fut.result()
        veredictos[quien][idx] = v
        if motivo:
            motivos[f"{quien}: {motivo}"] += 1
print("fallos del juez tras 5 intentos (se excluyen):", dict(motivos) or "ninguno")


900 preguntas, 714 con estructura valida para el juez...


fallos del juez tras 5 intentos (se excluyen): ninguno


In [5]:
def limpia(v):
    # La misma definicion de "sin ningun defecto" del notebook 03.
    return bool(v and not v.get("_estructura_rota") and v.get("respaldo")
                and not v.get("distractor_verdadero") and v.get("unica_respuesta")
                and v.get("tema_correcto"))


ETIQUETAS = {"maestro": "maestro (gpt-4o-mini)", **{k: v[0] for k, v in VARIANTES.items()}}
filas = []
for quien in QUIENES:
    todas = [v for v in veredictos[quien].values() if v is not None]
    juzgadas = [v for v in todas if not v.get("_estructura_rota")]
    n = max(len(juzgadas), 1)
    filas.append({
        "modelo": ETIQUETAS[quien],
        "estructura valida": f"{len(juzgadas)}/{len(todas)}",
        "correcta respaldada": f"{sum(bool(v.get('respaldo')) for v in juzgadas)/n*100:.1f}%",
        "sin distractor cierto": f"{sum(not v.get('distractor_verdadero', True) for v in juzgadas)/n*100:.1f}%",
        "una sola respuesta": f"{sum(bool(v.get('unica_respuesta')) for v in juzgadas)/n*100:.1f}%",
        "tema correcto": f"{sum(bool(v.get('tema_correcto')) for v in juzgadas)/n*100:.1f}%",
        "SIN DEFECTO (de las juzgadas)": f"{sum(limpia(v) for v in juzgadas)/n*100:.1f}%",
        "SIN DEFECTO (de los fragmentos)": f"{sum(limpia(v) for v in todas)/max(len(todas),1)*100:.1f}%",
    })

calidad = pd.DataFrame(filas).set_index("modelo")
print(calidad.T.to_string())


modelo                          maestro (gpt-4o-mini) afinado (LoRA) base + few-shot base A: mismo prompt base B: + esquema base C: prompt del maestro
estructura valida                             150/150        150/150         146/150                0/150           143/150                    125/150
correcta respaldada                             97.3%          95.3%           98.6%                 0.0%             95.8%                      92.8%
sin distractor cierto                           96.7%          94.7%           97.3%                 0.0%             96.5%                      88.8%
una sola respuesta                              96.0%          93.3%           97.3%                 0.0%             96.5%                      87.2%
tema correcto                                   82.7%          82.7%           75.3%                 0.0%             81.1%                      76.0%
SIN DEFECTO (de las juzgadas)                   80.0%          76.7%           71.9%          

### Cómo leer la tabla

Hay dos filas de «sin defecto», y la diferencia pesa mucho en las variantes que
rompen la estructura:

- **de las juzgadas** solo cuenta las preguntas con estructura válida. Es la cifra
  comparable con el notebook 03, pero favorece a quien rompe muchas: si una
  variante solo produce 10 preguntas válidas y las 10 salen bien, da 100%.
- **de los fragmentos** cuenta como fallo cada estructura rota. Es la tasa que vive
  el usuario: de cada fragmento, ¿sale una pregunta aprovechable? **Es la que hay
  que mirar para comparar variantes.**

### La prueba pareada

Como todos respondieron sobre **los mismos** fragmentos, lo que decide es en
cuántos acierta uno y falla el otro: los pares discordantes. La prueba de McNemar
pregunta si esos discordantes se reparten de forma tan desigual que no puede ser
azar. Aquí se compara siempre **el afinado contra cada una de las otras fuentes**,
contando la estructura rota como fallo.


In [6]:
from scipy.stats import binomtest

validos = [i for i in range(len(casos))
           if all(veredictos[q].get(i) is not None for q in QUIENES)]
ok = {q: {i: limpia(veredictos[q][i]) for i in validos} for q in QUIENES}

filas_p = []
for otro in [q for q in QUIENES if q != "afinado"]:
    solo_af = sum(ok["afinado"][i] and not ok[otro][i] for i in validos)
    solo_otro = sum(ok[otro][i] and not ok["afinado"][i] for i in validos)
    p = binomtest(solo_af, solo_af + solo_otro, 0.5).pvalue if solo_af + solo_otro else 1.0
    filas_p.append({"afinado contra": ETIQUETAS[otro],
                    "solo acierta el afinado": solo_af, "solo acierta el otro": solo_otro,
                    "p (McNemar exacto)": round(p, 4)})

pareadas = pd.DataFrame(filas_p).set_index("afinado contra")
print(f"fragmentos con los seis veredictos: {len(validos)}\n")
print(pareadas.to_string())
print("\np < 0.05: la diferencia difícilmente es azar. p alto: no se distingue del ruido.")


fragmentos con los seis veredictos: 150

                            solo acierta el afinado  solo acierta el otro  p (McNemar exacto)
afinado contra                                                                               
maestro (gpt-4o-mini)                             8                    13              0.3833
base + few-shot                                  22                    12              0.1214
base A: mismo prompt                            115                     0              0.0000
base B: + esquema                                20                    14              0.3915
base C: prompt del maestro                       40                    10              0.0000

p < 0.05: la diferencia difícilmente es azar. p alto: no se distingue del ruido.


### Criterio por criterio

Si el total no muestra diferencia, puede que un criterio concreto sí la tenga y
quede diluido. Aquí solo entran los fragmentos donde **las dos** preguntas se
pudieron juzgar, así que mide la calidad del contenido sin mezclarla con la
estructura.


In [7]:
real = lambda v: v is not None and not v.get("_estructura_rota")
BUENO = {"respaldo": True, "distractor_verdadero": False,
         "unica_respuesta": True, "tema_correcto": True}

filas_c = []
for otro in ("fewshot", "A", "B", "C"):
    pares = [(veredictos[otro].get(i), veredictos["afinado"].get(i)) for i in range(len(casos))]
    pares = [(o, a) for o, a in pares if real(o) and real(a)]
    fila = {"base": ETIQUETAS[otro], "pares juzgados": len(pares)}
    for criterio, bueno in BUENO.items():
        if bueno:
            cumple = lambda v: bool(v.get(criterio))
        else:
            cumple = lambda v: v.get(criterio) is False
        solo_af = sum(cumple(a) and not cumple(o) for o, a in pares)
        solo_o = sum(cumple(o) and not cumple(a) for o, a in pares)
        p = binomtest(solo_af, solo_af + solo_o, 0.5).pvalue if solo_af + solo_o else 1.0
        fila[criterio] = f"{solo_af} contra {solo_o} (p={p:.2f})"
    filas_c.append(fila)

criterios = pd.DataFrame(filas_c).set_index("base")
print("cada celda: fragmentos donde solo cumple el afinado contra donde solo cumple el base (p)\n")
print(criterios.T.to_string())


cada celda: fragmentos donde solo cumple el afinado contra donde solo cumple el base (p)

base                       base + few-shot base A: mismo prompt     base B: + esquema base C: prompt del maestro
pares juzgados                         146                    0                   143                        125
respaldo               1 contra 6 (p=0.12)  0 contra 0 (p=1.00)   4 contra 5 (p=1.00)        5 contra 3 (p=0.73)
distractor_verdadero   4 contra 8 (p=0.39)  0 contra 0 (p=1.00)   4 contra 7 (p=0.55)       13 contra 5 (p=0.10)
unica_respuesta       4 contra 10 (p=0.18)  0 contra 0 (p=1.00)   4 contra 9 (p=0.27)       13 contra 5 (p=0.10)
tema_correcto         15 contra 5 (p=0.04)  0 contra 0 (p=1.00)  12 contra 9 (p=0.66)       12 contra 6 (p=0.24)


In [8]:
forma.to_csv(DATA / "comparacion_prompts_forma.csv")
calidad.to_csv(DATA / "comparacion_prompts_calidad.csv")
pareadas.to_csv(DATA / "comparacion_prompts_pareadas.csv")

detalle = []
for i, fila in casos.iterrows():
    reg = {"chunk_uid": fila["chunk_uid"], "tema": fila["question_focus"],
           "fragmento": fila["chunk_text"],
           "maestro": {"pregunta": fila["pregunta"], "correcta": fila["correcta"],
                       "veredicto": veredictos["maestro"].get(i)}}
    for clave in VARIANTES:
        reg[clave] = {"salida": dicts[clave][i] or salidas[clave][i],
                      "veredicto": veredictos[clave].get(i)}
    detalle.append(reg)
json.dump(detalle, open(DATA / "comparacion_prompts_generaciones.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=1)
print("guardado: comparacion_prompts_{forma,calidad,pareadas}.csv y _generaciones.json")


guardado: comparacion_prompts_{forma,calidad,pareadas}.csv y _generaciones.json


### Cada fuente contra el maestro

La prueba anterior compara todo contra el afinado. Esta responde otra pregunta:
**¿quién alcanza al maestro?** Si el afinado no se distingue del maestro y las
variantes del base sí, el empate con el maestro dice algo del entrenamiento; si
todas empatan, no.


In [1]:
# Cada fuente contra el maestro, fragmento a fragmento. Lee lo guardado arriba:
# no regenera nada.
import json
from pathlib import Path
from scipy.stats import binomtest

DATA = Path("../data")
detalle = json.load(open(DATA / "comparacion_prompts_generaciones.json", encoding="utf-8"))


def limpia(v):
    return bool(v and not v.get("_estructura_rota") and v.get("respaldo")
                and not v.get("distractor_verdadero") and v.get("unica_respuesta")
                and v.get("tema_correcto"))


ETIQ = {"afinado": "afinado (LoRA)", "fewshot": "base + few-shot",
        "A": "base A: mismo prompt", "B": "base B: + esquema",
        "C": "base C: prompt del maestro"}
print(f"{'contra el maestro':<28}{'sin defecto':>12}{'solo esta':>11}{'solo maestro':>14}{'p':>9}")
for clave, etiqueta in ETIQ.items():
    pares = [(x["maestro"]["veredicto"], x[clave]["veredicto"]) for x in detalle
             if x["maestro"]["veredicto"] is not None and x[clave]["veredicto"] is not None]
    solo_esta = sum(limpia(v) and not limpia(m) for m, v in pares)
    solo_m = sum(limpia(m) and not limpia(v) for m, v in pares)
    p = binomtest(solo_esta, solo_esta + solo_m, 0.5).pvalue if solo_esta + solo_m else 1.0
    tasa = sum(limpia(v) for m, v in pares) / len(pares) * 100
    print(f"{etiqueta:<28}{tasa:>11.1f}%{solo_esta:>11}{solo_m:>14}{p:>9.4f}")
print("\np < 0.05: se distingue del maestro. p alto: no se distingue del ruido.")


contra el maestro            sin defecto  solo esta  solo maestro        p
afinado (LoRA)                     76.7%          8            13   0.3833
base + few-shot                    70.0%          9            24   0.0135
base A: mismo prompt                0.0%          0           120   0.0000
base B: + esquema                  72.7%         13            24   0.0989
base C: prompt del maestro         56.7%          7            42   0.0000

p < 0.05: se distingue del maestro. p alto: no se distingue del ruido.


### ¿Por qué se rompe la estructura?

Clasifica cada salida con la estructura rota. Descarta dos objeciones: que las
roturas sean respuestas cortadas por el límite de 300 tokens, y que las listas con
cuatro incorrectas se pudieran rescatar quitando la respuesta correcta repetida.


In [1]:
# Por que se rompe la estructura: clasifica cada salida rota. Lee lo guardado
# arriba: no regenera nada.
import json, unicodedata
from collections import Counter
from pathlib import Path

DATA = Path("../data")
detalle = json.load(open(DATA / "comparacion_prompts_generaciones.json", encoding="utf-8"))
norm = lambda s: " ".join("".join(c for c in unicodedata.normalize("NFD", str(s).lower())
                                  if unicodedata.category(c) != "Mn").strip(" .").split())


def estructura_ok(d):
    return bool(isinstance(d, dict) and d.get("pregunta") and d.get("correcta")
                and isinstance(d.get("incorrectas"), list) and len(d["incorrectas"]) == 3)


def motivo(s):
    if isinstance(s, dict):
        if s.get("apto") is False:
            return "descarto el fragmento (apto=false)"
        inc = s.get("incorrectas")
        if isinstance(inc, list) and len(inc) == 4:
            repetida = norm(s.get("correcta")) in [norm(x) for x in inc]
            return "4 incorrectas, " + ("una es la correcta repetida" if repetida
                                        else "ninguna es la correcta: no se puede rescatar")
        if isinstance(inc, list):
            return f"{len(inc)} incorrecta(s)"
        return f"otras claves: {sorted(s)}"
    txt = str(s).strip()
    return "JSON invalido, " + ("cerrado con }: no es un corte" if txt.endswith("}")
                                else "sin cerrar: posible corte por limite de tokens")


for clave, etiqueta in (("afinado", "afinado (LoRA)"), ("fewshot", "base + few-shot"),
                        ("A", "base A: mismo prompt"), ("B", "base B: + esquema"),
                        ("C", "base C: prompt del maestro")):
    rotas = Counter(motivo(x[clave]["salida"]) for x in detalle
                    if not estructura_ok(x[clave]["salida"]))
    print(f"{etiqueta}: {sum(rotas.values())} rotas de {len(detalle)}")
    for m, n in rotas.most_common():
        print(f"   {n:>3}  {m}")


afinado (LoRA): 0 rotas de 150
base + few-shot: 4 rotas de 150
     4  4 incorrectas, ninguna es la correcta: no se puede rescatar
base A: mismo prompt: 150 rotas de 150
   150  otras claves: ['opciones', 'pregunta', 'respuesta_correcta']
base B: + esquema: 7 rotas de 150
     7  4 incorrectas, ninguna es la correcta: no se puede rescatar
base C: prompt del maestro: 25 rotas de 150
     8  1 incorrecta(s)
     7  4 incorrectas, ninguna es la correcta: no se puede rescatar
     7  JSON invalido, cerrado con }: no es un corte
     3  descarto el fragmento (apto=false)


## B. ¿La robustez del formato es mérito del fine-tuning?

El notebook 04 mostró que el modelo afinado da 60 de 60 JSON válidos en todas las
temperaturas, incluida 1.0, y se atribuyó eso al fine-tuning. **Pero ese barrido
corrió solo con el afinado**: no hubo control con el base.

Aquí se hace el control, con el base y su few-shot. Los **mismos 20 fragmentos**
del notebook 04 (`random_state=42`), 3 repeticiones y las mismas temperaturas,
con semilla fija para que sea reproducible.


In [9]:
casos_b = test_df.sample(20, random_state=42)       # los mismos del notebook 04
frag_b = casos_b["chunk_text"].tolist()
REPS = 3
construir_fs = VARIANTES["fewshot"][2]

filas_b = []
for temp in (0.7, 1.0):
    torch.manual_seed(SEMILLA)
    prompts = [construir_fs(f) for f in frag_b] * REPS   # rep 1, rep 2, rep 3
    textos, _, truncadas = generar(prompts, usar_base=True, do_sample=True,
                                   temperature=temp, top_p=0.9, top_k=50)
    ds = [parsear(t) for t in textos]
    variedad = []
    for j in range(len(frag_b)):
        preguntas = {sin_acentos(ds[j + k * len(frag_b)]["pregunta"]).strip()
                     for k in range(REPS) if estructura_ok(ds[j + k * len(frag_b)])}
        variedad.append(len(preguntas))
    total = len(textos)
    filas_b.append({"modelo": "base + few-shot", "configuracion": f"temp {temp}",
                    "JSON valido": f"{sum(d is not None for d in ds)}/{total}",
                    "estructura": f"{sum(estructura_ok(d) for d in ds)}/{total}",
                    "variedad": round(sum(variedad) / len(variedad), 2),
                    "truncadas": truncadas})

# Los resultados del afinado, tal como los midio el notebook 04.
nb04 = pd.read_csv(DATA / "barrido_generacion.csv")
for conf in ("temp 0.7", "temp 1.0"):
    f = nb04[nb04["configuracion"] == conf].iloc[0]
    filas_b.append({"modelo": "afinado (notebook 04)", "configuracion": conf,
                    "JSON valido": f["JSON valido"], "estructura": f["estructura"],
                    "variedad": f["variedad"], "truncadas": f["truncadas"]})

control = pd.DataFrame(filas_b).sort_values(["configuracion", "modelo"]).reset_index(drop=True)
print(control.to_string(index=False))
control.to_csv(DATA / "comparacion_prompts_control_formato.csv", index=False)


               modelo configuracion JSON valido estructura  variedad  truncadas
afinado (notebook 04)      temp 0.7       60/60      60/60      2.30          0
      base + few-shot      temp 0.7       60/60      58/60      2.40          0
afinado (notebook 04)      temp 1.0       60/60      60/60      2.70          0
      base + few-shot      temp 1.0       60/60      58/60      2.55          0


## Conclusiones

### 1. Frente al prompt que se usaría sin entrenar, el afinado gana con claridad

Darle al base las mismas instrucciones que recibió el maestro (variante C) da
**56.7% sin defecto contra 76.7% del afinado, p < 0.0001**. Las reglas que
`gpt-4o-mini` sigue sin problema desbordan al modelo de 4B: rompe 25 estructuras
de 150 —ninguna por corte de tokens— y sus preguntas válidas también son peores
(68.0% contra 76.7%). **El fine-tuning incorporó las reglas del maestro al modelo;
el base no logra aplicarlas leyéndolas en el prompt.**

### 2. El fine-tuning le enseñó el formato

Con el mismo prompt que el afinado (variante A), el base inventa sus propias
claves —`opciones`, `respuesta_correcta`— en las 150 respuestas. Y ninguna
variante del base mantiene la estructura siempre: el few-shot rompe 4, B rompe 7
y C rompe 25 de 150; el afinado, ninguna. Las listas con cuatro incorrectas no se
pueden rescatar: en ningún caso la cuarta es la correcta repetida.

### 3. El afinado alcanza al maestro; el base con few-shot no

Contra el maestro, el afinado no se distingue (8 contra 13, p = 0.38). El base con
few-shot sí queda por debajo (9 contra 24, p = 0.014), y la diferencia sobrevive a
la corrección de Holm por las cinco comparaciones contra el maestro.

**Pero la comparación directa entre afinado y few-shot no es significativa**
(76.7% contra 70.0%, p = 0.12). Que uno alcance al maestro y el otro no, no
equivale a que uno supere al otro. En la versión anterior de este notebook, con
24 veredictos perdidos por fallos de la API, el few-shot tampoco se distinguía del
maestro (p = 0.22).

### 4. Frente a un prompt mínimo con el esquema, empate en contenido

La variante B —la instrucción corta más el esquema, 187 tokens— da 72.7% contra
76.7%, p = 0.39, y tampoco se distingue del maestro (p = 0.10). **El «prompt cinco
veces más corto» es cierto frente al few-shot (695 tokens) y frente a las reglas
del maestro (748), pero no frente a B**, donde la diferencia es 137 contra 187
tokens. Lo que B no iguala es la estructura: rompe 7 de 150.

### 5. La robustez al muestreo no es mérito del fine-tuning

El base con few-shot da 58 de 60 estructuras válidas a temperatura 0.7 y a 1.0,
con variedad parecida a la del afinado. La diferencia es la misma de siempre: dos
estructuras rotas de cada 60.

### El costo: velocidad

El afinado es el más lento: 0.97 s por pregunta, contra 0.76-0.85 s de las
variantes del base. La explicación probable es que el LoRA sin fusionar agrega
cómputo en cada capa; no se midió.

### Una cifra que no hay que usar

En «tema correcto» el afinado supera al few-shot con p = 0.04. Es una de 16
pruebas criterio por criterio, no sobrevive a ninguna corrección por
comparaciones múltiples, y en la versión anterior de este notebook la misma
comparación dio p = 0.31.

### En una línea

El fine-tuning incorporó al modelo el formato y las reglas del maestro. Frente a
darle esas reglas por escrito, el afinado es claramente mejor y usa un prompt cinco
veces más corto; frente a un prompt mínimo bien diseñado empata en contenido, pero
es la única variante del modelo local que nunca rompe el formato.
